# Notebook 05 — Sequence Topology Drift

**Repo:** `residual-phase-lock`  
**Notebook:** `05_sequence_topology_drift.ipynb`

## Claim

> Sequence fit does not guarantee structure preservation.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence-style models drift from global structure
```

This notebook uses balanced parentheses as a minimal sequence-topology task. Local token statistics can look plausible while global nesting structure still drifts.

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split

# Works in GitHub Actions from repo root.
# Works in Colab if notebook is opened from the GitHub repo.
# Fallback helps if Colab starts inside notebooks/.
if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(46)
random.seed(46)

NOTEBOOK_ID = "05"
NOTEBOOK_SLUG = "sequence_topology_drift"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Define global structure

A parentheses string is globally valid if:

1. final balance is zero,
2. every prefix balance is nonnegative.

Example:

```text
(()())   valid
())(()   invalid
```

The second string can share many local token patterns with valid strings while still violating global nesting.

In [ ]:
def is_balanced(seq):
    balance = 0
    min_prefix = 0
    for ch in seq:
        balance += 1 if ch == "(" else -1
        min_prefix = min(min_prefix, balance)
    return int(balance == 0 and min_prefix >= 0)

def sequence_stats(seq):
    balance = 0
    min_prefix = 0
    max_prefix = 0
    balances = []
    for ch in seq:
        balance += 1 if ch == "(" else -1
        balances.append(balance)
        min_prefix = min(min_prefix, balance)
        max_prefix = max(max_prefix, balance)

    return {
        "length": len(seq),
        "open_count": seq.count("("),
        "close_count": seq.count(")"),
        "final_balance": balance,
        "min_prefix_balance": min_prefix,
        "max_prefix_balance": max_prefix,
        "first_is_open": int(seq[0] == "("),
        "last_is_close": int(seq[-1] == ")"),
        "valid": is_balanced(seq),
    }

def random_balanced_sequence(n_pairs):
    # Random Catalan-like construction by shuffling choices while preserving validity.
    seq = []
    open_used = 0
    close_used = 0

    while len(seq) < 2 * n_pairs:
        choices = []
        if open_used < n_pairs:
            choices.append("(")
        if close_used < open_used:
            choices.append(")")
        ch = random.choice(choices)
        seq.append(ch)
        if ch == "(":
            open_used += 1
        else:
            close_used += 1

    return "".join(seq)

def corrupt_sequence(seq, n_swaps=2):
    chars = list(seq)
    for _ in range(n_swaps):
        i, j = random.sample(range(len(chars)), 2)
        chars[i], chars[j] = chars[j], chars[i]
    return "".join(chars)

print(is_balanced("(()())"), is_balanced("())(()"))

## 3. Generate sequence data

We create valid balanced sequences and corrupted / random invalid sequences.

The model will only receive local summary features, not an explicit stack machine.

In [ ]:
def make_sequence_dataset(n_valid=2500, n_invalid=2500, min_pairs=4, max_pairs=12):
    rows = []

    # Valid examples
    for _ in range(n_valid):
        n_pairs = random.randint(min_pairs, max_pairs)
        seq = random_balanced_sequence(n_pairs)
        row = sequence_stats(seq)
        row["sequence"] = seq
        row["source"] = "valid_generated"
        rows.append(row)

    # Invalid examples: corrupted balanced + random strings
    for i in range(n_invalid):
        n_pairs = random.randint(min_pairs, max_pairs)

        if i % 2 == 0:
            seq = random_balanced_sequence(n_pairs)
            seq = corrupt_sequence(seq, n_swaps=random.randint(1, 4))
        else:
            seq = "".join(random.choice(["(", ")"]) for _ in range(2 * n_pairs))

        if is_balanced(seq) == 1:
            seq = ")" + seq[1:]  # force invalid if randomness accidentally balanced

        row = sequence_stats(seq)
        row["sequence"] = seq
        row["source"] = "invalid_generated"
        rows.append(row)

    return pd.DataFrame(rows)

df = make_sequence_dataset()

# Raw generated data is useful for inspection but optional to commit.
exp.save_csv(df, "sequence_dataset")

df.head()

## 4. Local features only

The classifier receives local / shallow features:

```text
length, token counts, first token, last token, bigram counts
```

We deliberately exclude the true global checks:

```text
final_balance
min_prefix_balance
valid
```

Those are used later for drift analysis.

In [ ]:
def bigram_counts(seq):
    bigrams = {"((": 0, "()": 0, ")(": 0, "))": 0}
    for a, b in zip(seq[:-1], seq[1:]):
        bigrams[a + b] += 1
    return bigrams

bigram_df = pd.DataFrame([bigram_counts(seq) for seq in df["sequence"]])

feature_df = pd.concat([
    df[["length", "open_count", "close_count", "first_is_open", "last_is_close"]].reset_index(drop=True),
    bigram_df.reset_index(drop=True),
], axis=1)

target = df["valid"].values

feature_df.head()

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test = train_test_split(
    feature_df,
    target,
    df,
    test_size=0.35,
    random_state=46,
    stratify=target,
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_accuracy = float(accuracy_score(y_train, y_train_pred))
test_accuracy = float(accuracy_score(y_test, y_test_pred))

print(f"Train accuracy: {train_accuracy:.4f}")
print(f"Test accuracy:  {test_accuracy:.4f}")

## 5. Measure sequence topology drift

Topology drift occurs whenever local features lead to a prediction that disagrees with the global nesting rule.

```text
drift = predicted_validity ≠ true_validity
```

In [ ]:
test_result = df_test.copy().reset_index(drop=True)
test_result["predicted_valid"] = y_test_pred
test_result["residual"] = test_result["valid"] - test_result["predicted_valid"]
test_result["drift"] = (test_result["predicted_valid"] != test_result["valid"]).astype(int)

test_drift_rate = float(test_result["drift"].mean())
train_drift_rate = float((y_train_pred != y_train).mean())
generalization_gap = float(train_accuracy - test_accuracy)

print(f"Train drift rate: {train_drift_rate:.4f}")
print(f"Test drift rate:  {test_drift_rate:.4f}")

# Explanatory drift table, not full raw training data.
example_sequences = pd.concat([
    test_result[test_result["drift"] == 1].head(8),
    test_result[test_result["drift"] == 0].head(8),
], axis=0)[[
    "sequence",
    "valid",
    "predicted_valid",
    "residual",
    "drift",
    "final_balance",
    "min_prefix_balance",
    "source",
]]

exp.save_csv(example_sequences, "example_sequences")
example_sequences

## 6. Accuracy vs drift

Accuracy gives a local performance summary. Drift identifies structural disagreement.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "train_accuracy",
        "test_accuracy",
        "train_drift_rate",
        "test_drift_rate",
        "generalization_gap",
    ],
    "value": [
        train_accuracy,
        test_accuracy,
        train_drift_rate,
        test_drift_rate,
        generalization_gap,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

In [ ]:
plt.figure(figsize=(7, 4))
labels = ["Train accuracy", "Test accuracy", "Train drift", "Test drift"]
values = [train_accuracy, test_accuracy, train_drift_rate, test_drift_rate]
plt.bar(labels, values)
plt.ylim(0, 1)
plt.ylabel("rate")
plt.title("Sequence fit and topology drift")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
exp.save_fig("accuracy_vs_drift")
plt.show()

## 7. Drift by minimum prefix balance

Minimum prefix balance is a global structure signal.  
Negative values indicate that the sequence closed more parentheses than it had opened at some prefix.

If drift clusters by this variable, the model is not merely making random errors. It is drifting from global topology.

In [ ]:
drift_by_prefix = (
    test_result.groupby("min_prefix_balance")
    .agg(
        count=("drift", "size"),
        drift_rate=("drift", "mean"),
        valid_rate=("valid", "mean"),
        mean_residual=("residual", "mean"),
    )
    .reset_index()
    .sort_values("min_prefix_balance")
)

exp.save_csv(drift_by_prefix, "drift_by_min_prefix_balance")
drift_by_prefix.head(10)

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    drift_by_prefix["min_prefix_balance"],
    drift_by_prefix["drift_rate"],
    marker="o",
)
plt.ylim(0, 1)
plt.xlabel("minimum prefix balance")
plt.ylabel("drift rate")
plt.title("Topology drift by minimum prefix balance")
plt.grid(True, alpha=0.3)
plt.tight_layout()
exp.save_fig("drift_by_min_prefix_balance")
plt.show()

## 8. Residual distribution

The residual encodes the direction of validity error:

```text
+1 → true valid, predicted invalid
-1 → true invalid, predicted valid
0  → no drift
```

In [ ]:
residual_counts = (
    test_result["residual"]
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="count")
)

exp.save_csv(residual_counts, "residual_counts")

plt.figure(figsize=(6, 4))
plt.bar(residual_counts["residual"].astype(str), residual_counts["count"])
plt.xlabel("residual")
plt.ylabel("count")
plt.title("Residual distribution for sequence topology drift")
plt.tight_layout()
exp.save_fig("residual_distribution")
plt.show()

residual_counts

## 9. Confusion matrix

The confusion matrix shows drift between globally valid and invalid states.

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)
cm_df = pd.DataFrame(cm, index=["true_invalid", "true_valid"], columns=["pred_invalid", "pred_valid"])
exp.save_csv(cm_df.reset_index().rename(columns={"index": "label"}), "confusion_matrix")

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Sequence validity confusion matrix")
plt.xlabel("Predicted validity")
plt.ylabel("True validity")
plt.xticks([0, 1], ["invalid", "valid"])
plt.yticks([0, 1], ["invalid", "valid"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.colorbar()
plt.tight_layout()
exp.save_fig("confusion_matrix")
plt.show()

cm_df

## 10. Sequence examples

We save a small table of drift and non-drift examples for the markdown doc and paper notes.

In [ ]:
display_examples = example_sequences.copy()
display_examples["short_sequence"] = display_examples["sequence"].str.slice(0, 40)
display_examples

## 11. Generate markdown summary

This writes:

```text
docs/05_sequence_topology_drift.md
```

In [ ]:
exp.write_md(
    title="Sequence Topology Drift",
    metrics_dict={
        "Train accuracy": train_accuracy,
        "Test accuracy": test_accuracy,
        "Train drift rate": train_drift_rate,
        "Test drift rate": test_drift_rate,
        "Generalization gap": generalization_gap,
    },
    figure_names=[
        "accuracy_vs_drift",
        "drift_by_min_prefix_balance",
        "residual_distribution",
        "confusion_matrix",
    ],
    interpretation="""
sequence fit ≠ structure preservation
local token features can drift from global nesting
residuals expose sequence topology drift
""",
)

## 12. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
05_sequence_topology_drift_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "05_sequence_topology_drift_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 13. Takeaway

This notebook supports the sequence-model bridge:

```text
sequence fit ≠ structure preservation
local token features can drift from global structure
residuals expose topology drift
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → known constraint corrects drift
04 → learned residual corrects drift
05 → sequence topology drift connects to transformer motivation
```

Suggested next notebook:

```text
06_sequence_phase_lock_correction.ipynb
```